In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/clickstream_clean.csv")

In [13]:
print(df.shape)
df.head()

(165474, 19)


,year,month,day,order,country,session_id,main_category,product_id,colour,location,photo_type,price,price_category,page_number,main_category_name,photo_type_name,location_name,price_category_name,colour_name
0,2008,4,1,1,29,1,1,A13,1,5,1,28,2,1,Trousers,Front View,Bottom Middle,Below Category Average,Beige
1,2008,4,1,2,29,1,1,A16,1,6,1,33,2,1,Trousers,Front View,Bottom Right,Below Category Average,Beige
2,2008,4,1,3,29,1,2,B4,10,2,1,52,1,1,Skirts,Front View,Top Middle,Above Category Average,Violet
3,2008,4,1,4,29,1,2,B17,6,6,2,38,2,1,Skirts,Profile View,Bottom Right,Below Category Average,Gray
4,2008,4,1,5,29,1,2,B8,4,3,2,52,1,1,Skirts,Profile View,Top Right,Above Category Average,Brown


# Session-Level Feature Engineering

## Objective

The raw clickstream dataset records one row per click event.

However, product and business decisions are typically made at the customer-session level rather than the individual click level.

The objective of this notebook is to transform click-level data into a session-level dataset by engineering meaningful engagement metrics that can later be used for product analytics, statistical analysis, and experiment recommendations.

In [14]:
session_summary = (
    df.groupby("session_id")
      .agg(
          click_count=("order", "count"),
          unique_products=("product_id", "nunique"),
          unique_categories=("main_category", "nunique"),
          max_page=("page_number", "max"),
          average_price=("price", "mean"),
          max_order=("order", "max")
      )
      .reset_index()
)

In [15]:
session_summary.head()

,session_id,click_count,unique_products,unique_categories,max_page,average_price,max_order
0,1,9,9,4,5,42.111111,9
1,2,10,8,3,2,50.000000,10
2,3,6,6,3,5,42.166667,6
3,4,4,4,2,3,45.250000,4
4,5,1,1,1,2,57.000000,1


In [16]:
session_summary.describe()

,session_id,click_count,unique_products,unique_categories,max_page,average_price,max_order
count,24026.00000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000
mean,12013.50000,6.887289,6.174353,1.851661,2.250271,44.374906,6.887289
std,6935.85312,8.995161,7.525282,0.956479,1.284408,8.454850,8.995161
min,1.00000,1.000000,1.000000,1.000000,1.000000,18.000000,1.000000
25%,6007.25000,2.000000,2.000000,1.000000,1.000000,38.166667,2.000000
50%,12013.50000,4.000000,4.000000,2.000000,2.000000,43.666667,4.000000
75%,18019.75000,8.000000,8.000000,3.000000,3.000000,49.000000,8.000000
max,24026.00000,195.000000,168.000000,4.000000,5.000000,82.000000,195.000000


In [17]:
first_category = (
    df.sort_values(["session_id", "order"])
      .groupby("session_id")
      .first()["main_category"]
)

In [18]:
first_photo = (
    df.sort_values(["session_id", "order"])
      .groupby("session_id")
      .first()["photo_type"]
)

In [19]:
first_location = (
    df.sort_values(["session_id", "order"])
      .groupby("session_id")
      .first()["location"]
)

In [20]:
session_summary["first_category"] = session_summary["session_id"].map(first_category)

session_summary["first_photo"] = session_summary["session_id"].map(first_photo)

session_summary["first_location"] = session_summary["session_id"].map(first_location)

In [22]:
category_labels = {
    1: "Trousers",
    2: "Skirts",
    3: "Blouses",
    4: "Sale"
}

In [24]:
photo_labels = {
    1: "Front View",
    2: "Profile View"
}



In [25]:
location_labels = {
    1: "Top Left",
    2: "Top Middle",
    3: "Top Right",
    4: "Bottom Left",
    5: "Bottom Middle",
    6: "Bottom Right"
}

In [26]:
session_summary["first_category_name"] = (
    session_summary["first_category"].map(category_labels)
)

session_summary["first_photo_name"] = (
    session_summary["first_photo"].map(photo_labels)
)

session_summary["first_location_name"] = (
    session_summary["first_location"].map(location_labels)
)

In [27]:
session_summary["engagement_score"] = (
    session_summary["click_count"]
    + session_summary["unique_products"]
    + session_summary["unique_categories"]
    + session_summary["max_page"]
)

In [28]:
session_summary.to_csv(
    "../data/processed/session_summary.csv",
    index=False
)

In [29]:
session_summary.head(10)


,session_id,click_count,unique_products,unique_categories,max_page,average_price,max_order,first_category,first_photo,first_location,first_category_name,first_photo_name,first_location_name,engagement_score
0,1,9,9,4,5,42.111111,9,1,1,5,Trousers,Front View,Bottom Middle,27
1,2,10,8,3,2,50.000000,10,2,1,5,Skirts,Front View,Bottom Middle,23
2,3,6,6,3,5,42.166667,6,2,2,6,Skirts,Profile View,Bottom Right,20
3,4,4,4,2,3,45.250000,4,1,1,6,Trousers,Front View,Bottom Right,13
4,5,1,1,1,2,57.000000,1,3,1,1,Blouses,Front View,Top Left,5
5,6,5,5,2,3,42.800000,5,3,1,3,Blouses,Front View,Top Right,15
6,7,11,11,4,4,37.818182,11,1,1,4,Trousers,Front View,Bottom Left,30
7,8,9,9,2,3,37.888889,9,3,2,5,Blouses,Profile View,Bottom Middle,23
8,9,3,3,2,2,41.000000,3,1,1,6,Trousers,Front View,Bottom Right,10
9,10,5,5,2,1,45.400000,5,1,1,1,Trousers,Front View,Top Left,13


In [31]:
session_summary.head()
#session_summary.describe()
#session_summary.info()

,session_id,click_count,unique_products,unique_categories,max_page,average_price,max_order,first_category,first_photo,first_location,first_category_name,first_photo_name,first_location_name,engagement_score
0,1,9,9,4,5,42.111111,9,1,1,5,Trousers,Front View,Bottom Middle,27
1,2,10,8,3,2,50.000000,10,2,1,5,Skirts,Front View,Bottom Middle,23
2,3,6,6,3,5,42.166667,6,2,2,6,Skirts,Profile View,Bottom Right,20
3,4,4,4,2,3,45.250000,4,1,1,6,Trousers,Front View,Bottom Right,13
4,5,1,1,1,2,57.000000,1,3,1,1,Blouses,Front View,Top Left,5


In [32]:
session_summary.describe()

,session_id,click_count,unique_products,unique_categories,max_page,average_price,max_order,first_category,first_photo,first_location,engagement_score
count,24026.00000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000,24026.000000
mean,12013.50000,6.887289,6.174353,1.851661,2.250271,44.374906,6.887289,2.121618,1.225048,3.111005,17.163573
std,6935.85312,8.995161,7.525282,0.956479,1.284408,8.454850,8.995161,1.094583,0.417623,1.747791,17.651098
min,1.00000,1.000000,1.000000,1.000000,1.000000,18.000000,1.000000,1.000000,1.000000,1.000000,4.000000
25%,6007.25000,2.000000,2.000000,1.000000,1.000000,38.166667,2.000000,1.000000,1.000000,1.000000,6.000000
50%,12013.50000,4.000000,4.000000,2.000000,2.000000,43.666667,4.000000,2.000000,1.000000,3.000000,12.000000
75%,18019.75000,8.000000,8.000000,3.000000,3.000000,49.000000,8.000000,3.000000,1.000000,5.000000,21.000000
max,24026.00000,195.000000,168.000000,4.000000,5.000000,82.000000,195.000000,4.000000,2.000000,6.000000,372.000000


In [33]:
session_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24026 entries, 0 to 24025
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           24026 non-null  int64  
 1   click_count          24026 non-null  int64  
 2   unique_products      24026 non-null  int64  
 3   unique_categories    24026 non-null  int64  
 4   max_page             24026 non-null  int64  
 5   average_price        24026 non-null  float64
 6   max_order            24026 non-null  int64  
 7   first_category       24026 non-null  int64  
 8   first_photo          24026 non-null  int64  
 9   first_location       24026 non-null  int64  
 10  first_category_name  24026 non-null  object 
 11  first_photo_name     24026 non-null  object 
 12  first_location_name  24026 non-null  object 
 13  engagement_score     24026 non-null  int64  
dtypes: float64(1), int64(10), object(3)
memory usage: 2.6+ MB


In [ ]:
#we've moved from event-level data to customer-session-level data